# Day 39: Model Merging with MergeKit

MergeKit is a toolkit for merging pre-trained language models using an out-of-core approach that works even on resource-constrained hardware (CPU only or as little as 8GB VRAM).

In this notebook we'll create a YAML config for a linear (weighted average) merge and run it.

In [ ]:
import os
import yaml
from pathlib import Path

# Create a config directory
config_dir = Path("./merge_configs")
config_dir.mkdir(exist_ok=True)

## 1. Linear Merge (Weighted Average)

The simplest method: a weighted average of the model parameters.

```yaml
merge_method: linear
slices:
  - sources:
      - model: microsoft/phi-2
        layer_range: [0, 32]
        parameters:
          weight: 0.5
      - model: cognitivecomputations/dolphin-2_6-phi-2
        layer_range: [0, 32]
        parameters:
          weight: 0.5
dtype: float16
```

In [ ]:
linear_config = {
    "merge_method": "linear",
    "slices": [
        {
            "sources": [
                {
                    "model": "microsoft/phi-2",
                    "layer_range": [0, 32],
                    "parameters": {"weight": 0.5},
                },
                {
                    "model": "cognitivecomputations/dolphin-2_6-phi-2",
                    "layer_range": [0, 32],
                    "parameters": {"weight": 0.5},
                },
            ]
        }
    ],
    "dtype": "float16",
}

with open(config_dir / "linear_merge.yaml", "w") as f:
    yaml.dump(linear_config, f)
print("Config saved. To run the merge:")
print(f"mergekit-yaml {config_dir}/linear_merge.yaml ./merged_model --cuda")

## 2. SLERP Merge

Spherical Linear Interpolation is better suited for interpolating high‑dimensional vectors, preserving magnitude and curvature.

In [ ]:
slerp_config = {
    "merge_method": "slerp",
    "base_model": "microsoft/phi-2",
    "models": [
        {"model": "cognitivecomputations/dolphin-2_6-phi-2", "parameters": {"weight": 0.7}},
        {"model": "teknium/OpenHermes-2.5-Mistral-7B", "parameters": {"weight": 0.3}},
    ],
    "dtype": "float16",
}

with open(config_dir / "slerp_merge.yaml", "w") as f:
    yaml.dump(slerp_config, f)
print("Config saved.")

## 3. TIES Merge (Trim, Elect Sign & Merge)

TIES sparsifies task vectors to reduce interference between models. This often yields better results than linear or SLERP when merging models fine‑tuned for different tasks.

In [ ]:
ties_config = {
    "merge_method": "ties",
    "base_model": "microsoft/phi-2",
    "models": [
        {"model": "cognitivecomputations/dolphin-2_6-phi-2", "parameters": {"weight": 1.0, "density": 0.5}},
        {"model": "teknium/OpenHermes-2.5-Mistral-7B", "parameters": {"weight": 0.5, "density": 0.5}},
    ],
    "dtype": "float16",
}

with open(config_dir / "ties_merge.yaml", "w") as f:
    yaml.dump(ties_config, f)
print("Config saved.")

## 4. Running the Merge

Once you have your YAML file, you can run the merge using the `mergekit-yaml` command-line tool:

```bash
mergekit-yaml ./merge_configs/linear_merge.yaml ./output_model --cuda
```

The `--cuda` flag enables GPU acceleration, which is recommended for computationally intensive methods like SLERP and TIES.

In [ ]:
# Example: using Python subprocess to run the merge (optional)
import subprocess

def run_merge(config_path, output_dir, use_cuda=True):
    cmd = ["mergekit-yaml", config_path, output_dir]
    if use_cuda:
        cmd.append("--cuda")
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("Errors:", result.stderr)
    return result.returncode == 0

# Uncomment to run:
# run_merge(str(config_dir / "linear_merge.yaml"), "./phi_merged")

## 5. Testing the Merged Model

After merging, you can load the model using Transformers and test its capabilities.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def test_model(model_path, prompt):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=100)
    return tokenizer.decode(outputs[0])

# test_model("./phi_merged", "Explain quantum computing in simple terms:")
print("Model loading example – uncomment after merging.")